In [1]:
import pandas as pd
import numpy as np
import re

In [2]:
DATA_FOLDER = "./"

df = pd.read_parquet(DATA_FOLDER + "df_intermedio.parquet")
product_ids = pd.read_csv(DATA_FOLDER + "product_id_apredecir201912.txt", sep="\t")[
    "product_id"
].tolist()


In [3]:
df.drop(columns=["periodo_min_producto", "periodo_max_producto", "periodo_min_customer", "periodo_max_customer"], inplace=True, errors="ignore")
df

,product_id,customer_id,fecha,periodo,plan_precios_cuidados,cust_request_qty,cust_request_tn,tn,stock_final,cat1,cat2,cat3,brand,sku_size
0,20524,10234,2017-01,201701,0.0,2,0.05300,0.05300,NaN,HC,VAJILLA,Cristalino,Importado,500.0
1,20524,10234,2017-02,201702,0.0,0,0.00000,0.00000,NaN,HC,VAJILLA,Cristalino,Importado,500.0
2,20524,10234,2017-03,201703,0.0,1,0.01514,0.01514,NaN,HC,VAJILLA,Cristalino,Importado,500.0
3,20524,10234,2017-04,201704,0.0,0,0.00000,0.00000,NaN,HC,VAJILLA,Cristalino,Importado,500.0
4,20524,10234,2017-05,201705,0.0,0,0.00000,0.00000,NaN,HC,VAJILLA,Cristalino,Importado,500.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17173443,20770,10591,2019-12,201912,0.0,0,0.00000,0.00000,9.53225,HC,PROFESIONAL,LV ROPA POLVO,INDUSTRIAL,25.0
17173444,20770,10559,2019-12,201912,0.0,0,0.00000,0.00000,9.53225,HC,PROFESIONAL,LV ROPA POLVO,INDUSTRIAL,25.0
17173445,20770,10560,2019-12,201912,0.0,0,0.00000,0.00000,9.53225,HC,PROFESIONAL,LV ROPA POLVO,INDUSTRIAL,25.0
17173446,20770,10582,2019-12,201912,0.0,0,0.00000,0.00000,9.53225,HC,PROFESIONAL,LV ROPA POLVO,INDUSTRIAL,25.0


In [4]:
df["sku_size"]

0           500.0
1           500.0
2           500.0
3           500.0
4           500.0
            ...  
17173443     25.0
17173444     25.0
17173445     25.0
17173446     25.0
17173447     25.0
Name: sku_size, Length: 17173448, dtype: float64

In [5]:
cat_features = ["cat1", "cat2", "cat3", "brand"]
# replace nans with "unknown" in categorical features
for col in cat_features:
    # replace NaN values with "unknown"
    df[col] = df[col].fillna("unknown")
    

In [6]:
# replace sku_size nan with 0s
df["sku_size"] = df["sku_size"].fillna(0)
df["sku_size"].describe()

count    1.717345e+07
mean     4.739798e+02
std      8.867743e+02
min      0.000000e+00
25%      9.000000e+01
50%      2.400000e+02
75%      4.750000e+02
max      1.000000e+04
Name: sku_size, dtype: float64

In [7]:
# agrupo en 25 customers para que no tenga tantas filas

best_customers = (
    df.groupby('customer_id')['tn']
    .sum()
    .nlargest(5)
    .index
)
df_new = df[df['customer_id'].isin(best_customers)]

# Si es true creo un nuevo customer_id 0 que tiene la suma de los tn de los clientes que no estan en los mejores
df_others = df[~df['customer_id'].isin(best_customers)].copy()
df_others['customer_id'] = 0  # Agrupo los otros clientes bajo el ID 0
df_others = df_others.groupby(['product_id', "fecha"]).agg({
    "cust_request_qty": "sum",
    "cust_request_tn": "sum",
    "tn": "sum",
    "stock_final": "max",
    "cat1": "first",
    "cat2": "first",
    "cat3": "first",
    "brand": "first",
    "sku_size": "first",
    "customer_id": "first",
}
).reset_index()
df_new = pd.concat([df_new, df_others], ignore_index=True)
# ordeno el df por date_id, product y customer
df_new = df_new.sort_values(by=['fecha', 'product_id', 'customer_id'])

df = df_new

In [8]:
columns_to_scale = ["tn", "cust_request_qty", "cust_request_tn", "stock_final"]

# Paso 1: calcular el promedio de los primeros 20 valores no-cero para cada grupo y columna
def get_first_nz_mean(group, n=20):
    result = {}
    for col in columns_to_scale:
        non_zero_vals = group[col][group[col] != 0].iloc[:n]
        result[col] = non_zero_vals.mean() if not non_zero_vals.empty else 0
    return pd.Series(result)

scaler_df = df.groupby(['customer_id', 'product_id']).apply(get_first_nz_mean).reset_index()

# Paso 2: escalar cada grupo usando su valor promedio como referencia
def scale_group(group):
    key = group[['customer_id', 'product_id']].iloc[0]
    base = scaler_df[
        (scaler_df['customer_id'] == key['customer_id']) &
        (scaler_df['product_id'] == key['product_id'])
    ][columns_to_scale].iloc[0]

    group_scaled = group.copy()
    for col in columns_to_scale:
        scale = base[col]
        group_scaled[col] = group[col] / scale if scale != 0 else 0
    return group_scaled

df_scaled = df.groupby(['customer_id', 'product_id'], group_keys=False).apply(scale_group)

df_scaled

/tmp/ipykernel_435088/641134820.py:11: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  scaler_df = df.groupby(['customer_id', 'product_id']).apply(get_first_nz_mean).reset_index()
/tmp/ipykernel_435088/641134820.py:27: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_scaled = df.groupby(['customer_id', 'product_id'], group_keys=False).apply(scale_group)


,product_id,customer_id,fecha,periodo,plan_precios_cuidados,cust_request_qty,cust_request_tn,tn,stock_final,cat1,cat2,cat3,brand,sku_size
157610,20001,0,2017-01,NaN,NaN,1.109425,0.581189,0.599979,NaN,HC,ROPA LAVADO,Liquido,ARIEL,3000.0
67489,20001,10001,2017-01,201701.0,0.0,0.471092,0.742264,0.744614,NaN,HC,ROPA LAVADO,Liquido,ARIEL,3000.0
67417,20001,10002,2017-01,201701.0,0.0,0.712788,1.162225,1.123264,NaN,HC,ROPA LAVADO,Liquido,ARIEL,3000.0
67381,20001,10003,2017-01,201701.0,0.0,0.764045,1.180730,1.231587,NaN,HC,ROPA LAVADO,Liquido,ARIEL,3000.0
67453,20001,10004,2017-01,201701.0,0.0,1.077844,0.750660,0.785586,NaN,HC,ROPA LAVADO,Liquido,ARIEL,3000.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
152394,21276,10001,2019-12,201912.0,0.0,0.000000,0.000000,0.000000,0.859215,PC,PIEL1,Cara,NIVEA,140.0
152374,21276,10002,2019-12,201912.0,0.0,0.000000,0.000000,0.000000,0.859215,PC,PIEL1,Cara,NIVEA,140.0
152364,21276,10003,2019-12,201912.0,0.0,0.000000,0.000000,0.000000,0.859215,PC,PIEL1,Cara,NIVEA,140.0
152384,21276,10004,2019-12,201912.0,0.0,0.000000,0.000000,0.000000,0.859215,PC,PIEL1,Cara,NIVEA,140.0


In [9]:
df["stock_final"].describe()

count    82146.000000
mean        19.478147
std         55.625745
min        -27.311360
25%          1.160800
50%          5.419600
75%         17.586830
max       1562.024480
Name: stock_final, dtype: float64

In [10]:
# save scaler and df_scaled
scaler_df.to_parquet(DATA_FOLDER + "scaler_df.parquet", index=False)
df_scaled.to_parquet(DATA_FOLDER + "df_scaled.parquet", index=False)

In [12]:
# creo un nuevo dataset, llamado scaler_df que tiene product_id, customer_id y el primer valor distinto de 0 de cada columna que se debe escalar, por ejemplo columna tn_scaler
columns_to_scale = ["tn", "cust_request_qty", "cust_request_tn", "stock_final"]
scaler_df = pd.DataFrame(columns=["product_id", "customer_id"] + [col + "_scaler" for col in columns_to_scale])
for col in columns_to_scale:
    scaler_df[col + "_scaler"] = df.groupby(["product_id", "customer_id"])[col].transform(lambda x: x[x != 0].iloc[0] if not x[x != 0].empty else np.nan)
scaler_df

,product_id,customer_id,tn_scaler,cust_request_qty_scaler,cust_request_tn_scaler,stock_final_scaler
788050,NaN,NaN,139.028275,229.0,139.028275,NaN
337265,NaN,NaN,99.438606,11.0,99.438606,NaN
337013,NaN,NaN,35.728062,17.0,38.683010,NaN
336977,NaN,NaN,143.494263,17.0,143.494263,NaN
337049,NaN,NaN,184.729263,9.0,184.729263,NaN
...,...,...,...,...,...,...
761944,NaN,NaN,NaN,NaN,NaN,1.68932
761974,NaN,NaN,NaN,NaN,NaN,1.68932
761954,NaN,NaN,NaN,NaN,NaN,1.68932
761964,NaN,NaN,NaN,NaN,NaN,1.68932


In [3]:
target = df.groupby(["customer_id", "product_id"])["tn"].shift(-2)
# obtengo las 20 columnas con mayor correlación con el target
correlation = df[numeric_cols].corrwith(target).abs().sort_values(ascending=False)
top_20_cols = correlation.head(20).index.tolist()
print("Top 20 columns with highest correlation to target:")
print(top_20_cols)

/home/fede/programacion/labo3/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/home/fede/programacion/labo3/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


Top 20 columns with highest correlation to target:
['tn_wavelet_0_mean_lag_11', 'tn_wavelet_0_mean_lag_8', 'tn_wavelet_0_mean_lag_15', 'tn_wavelet_0_mean', 'tn_wavelet_0_mean_lag_2', 'tn_wavelet_0_mean_lag_1', 'tn_wavelet_0_mean_lag_3', 'tn_rolling_mean_12', 'tn_wavelet_0_mean_lag_6', 'tn_rolling_mean_12_lag_1', 'tn_wavelet_0_mean_lag_20', 'tn_rolling_mean_12_lag_2', 'tn_rolling_mean_12_lag_3', 'tn_rolling_mean_24', 'tn_rolling_mean_6_lag_6', 'tn_rolling_mean_24_lag_1', 'tn_wavelet_0_max_lag_11', 'tn_wavelet_0_max_lag_15', 'tn_wavelet_0_max', 'tn_wavelet_0_max_lag_2']


In [4]:
transformations = {
    "tn": [
        r"tn$",
        r"cust_request_qty_per_tn$",
        r"tn_lag_*",
        r"tn_rolling_mean_*",
        r"tn_rolling_max_*",
        r"tn_rolling_min_*",
        r"tn_.*_vendidas$",
        r"tn_agg*",
        r"tn_wavelet_*",
    ]
    + [r"stock_final$"]
    + [r"cust_request_tn_minus_tn$"]
    + [r"tn_diff_*"],
    "cust_request_qty": [
        r"cust_request_qty$",
        r"cust_request_qty_lag_*",
        r"cust_request_qty_rolling_mean_*",
        r"cust_request_qty_rolling_max_*",
        r"cust_request_qty_rolling_min_*",
        r"cust_request_qty_.*_vendidas$",
        r"cust_request_qty_agg*",
        r"cust_request_qty_wavelet_*",
    ]
    + [r"cust_request_qty_diff_*"],
}

# busco todas las columnas que empiezan con prod_ y agrego key y valor en transformation
for col in numeric_cols:
    if col.startswith("prod_"):
        transformations[col] = [r"{}$".format(col)]
transformations

{'tn': ['tn$',
  'cust_request_qty_per_tn$',
  'tn_lag_*',
  'tn_rolling_mean_*',
  'tn_rolling_max_*',
  'tn_rolling_min_*',
  'tn_.*_vendidas$',
  'tn_agg*',
  'tn_wavelet_*',
  'stock_final$',
  'cust_request_tn_minus_tn$',
  'tn_diff_*'],
 'cust_request_qty': ['cust_request_qty$',
  'cust_request_qty_lag_*',
  'cust_request_qty_rolling_mean_*',
  'cust_request_qty_rolling_max_*',
  'cust_request_qty_rolling_min_*',
  'cust_request_qty_.*_vendidas$',
  'cust_request_qty_agg*',
  'cust_request_qty_wavelet_*',
  'cust_request_qty_diff_*'],
 'prod_tn_wavelet_0_mean_lag_11_x_tn_wavelet_0_mean_lag_8': ['prod_tn_wavelet_0_mean_lag_11_x_tn_wavelet_0_mean_lag_8$'],
 'prod_tn_wavelet_0_mean_lag_11_x_tn_wavelet_0_mean_lag_15': ['prod_tn_wavelet_0_mean_lag_11_x_tn_wavelet_0_mean_lag_15$'],
 'prod_tn_wavelet_0_mean_lag_11_x_tn_wavelet_0_mean': ['prod_tn_wavelet_0_mean_lag_11_x_tn_wavelet_0_mean$'],
 'prod_tn_wavelet_0_mean_lag_11_x_tn_wavelet_0_mean_lag_2': ['prod_tn_wavelet_0_mean_lag_11_x_tn_

In [5]:
# hago el scaling que sera column/std(col_ref) para cada columna

# primero calculo el std por product_id y customer_id por cada key de transformation
# gruped_std tiene doble indice product_id y customer_id
prod_stats = df.groupby(["product_id", "customer_id"])[
    list(transformations.keys())
].agg(["std"])
prod_stats.columns = [
    f"{col[0]}_{col[1]}" for col in prod_stats.columns
]  # renombro las columnas para que no tengan tupla
prod_stats = prod_stats.reset_index()

df = df.set_index(['product_id', 'customer_id'])
prod_stats = prod_stats.set_index(['product_id', 'customer_id'])
# supress performance warnings
from pandas.errors import PerformanceWarning
import warnings
warnings.simplefilter(action="ignore", category=PerformanceWarning)

for trainer, regex_cols in transformations.items():
    for col in regex_cols:
        matching_cols = [c for c in numeric_cols if re.match(col, c)]
        if not matching_cols:
            continue
        print(f"Processing trainer: {trainer} with columns: {matching_cols}")
        for col in matching_cols:
            std_col = prod_stats[trainer + "_std"]
            # Alinear por índice, sin merge
            df[f"{col}_scaled"] = (df[col] / std_col).replace([np.inf, -np.inf], np.nan)
            # Opcional: fillna(0) si querés
            # df[f"{col}_scaled"] = df[f"{col}_scaled"].fillna(0)

# Si querés, reseteá el índice al final
df = df.reset_index()

Processing trainer: tn with columns: ['tn']
Processing trainer: tn with columns: ['tn_lag_1', 'tn_lag_2', 'tn_lag_3', 'tn_lag_6', 'tn_lag_8', 'tn_lag_11', 'tn_lag_15', 'tn_lag_20']
Processing trainer: tn with columns: ['tn_rolling_mean_6', 'tn_rolling_mean_12', 'tn_rolling_mean_24', 'tn_rolling_mean_12_lag_1', 'tn_rolling_mean_12_lag_2', 'tn_rolling_mean_12_lag_3', 'tn_rolling_mean_12_lag_6', 'tn_rolling_mean_12_lag_8', 'tn_rolling_mean_12_lag_11', 'tn_rolling_mean_12_lag_15', 'tn_rolling_mean_12_lag_20', 'tn_rolling_mean_24_lag_1', 'tn_rolling_mean_24_lag_2', 'tn_rolling_mean_24_lag_3', 'tn_rolling_mean_24_lag_6', 'tn_rolling_mean_24_lag_8', 'tn_rolling_mean_24_lag_11', 'tn_rolling_mean_6_lag_1', 'tn_rolling_mean_6_lag_2', 'tn_rolling_mean_6_lag_3', 'tn_rolling_mean_6_lag_6', 'tn_rolling_mean_6_lag_8', 'tn_rolling_mean_6_lag_11', 'tn_rolling_mean_6_lag_15', 'tn_rolling_mean_6_lag_20']
Processing trainer: tn with columns: ['tn_rolling_max_6', 'tn_rolling_max_12', 'tn_rolling_max_24', '

In [6]:
# creo el target
prod_stats.reset_index(inplace=True)
df["target"] = df.groupby(["product_id", "customer_id"])["tn"].shift(-2)
# elimino las rows donde target es nan
df = df[df["target"].notna()]
# hago el scaling del target por tn_std
df = df.merge(
    prod_stats[["product_id", "customer_id", "tn_std"]],
    on=["product_id", "customer_id"],
    how="left",
)
df["target_scaled"] = (df["target"] / df["tn_std"]).fillna(0)
# replace nan with 0
df.drop(columns=["tn_std"], inplace=True)


In [7]:
df["target_scaled"].describe()

count    755976.000000
mean          0.807325
std           1.069927
min           0.000000
25%           0.000000
50%           0.287944
75%           1.362039
max          20.666327
Name: target_scaled, dtype: float64

In [8]:
import numpy as np
import pandas as pd
from sklearn.model_selection import BaseCrossValidator

class CustomTimeSeriesSplitter(BaseCrossValidator):
    def __init__(self, subsample_prop=0.5, test_months=1, random_state=None):
        assert 0 < subsample_prop <= 1, "subsample_prop must be in (0, 1]"
        self.subsample_prop = subsample_prop
        self.test_months = test_months
        self.random_state = np.random.RandomState(random_state)

    def get_n_splits(self, X=None, y=None, groups=None):
        return self.test_months

    def split(self, X, y=None, groups=None):
        df = X.reset_index(drop=True).copy()
        max_date = df['date_id'].max()
        pair_col = ['product_id', 'customer_id']

        # Calcular tn total y asignar deciles
        total_tn = (
            df.groupby(pair_col)['tn'].sum()
            .reset_index(name='total_tn')
            .sort_values('total_tn', ascending=False)
            .reset_index(drop=True)
        )
        total_tn['quantile'] = pd.qcut(total_tn.index, 10, labels=False)

        # Samplear series por quantil
        sampled_series = []
        for q in range(10):
            group = total_tn[total_tn['quantile'] == q]
            n = max(1, int(len(group) * self.subsample_prop))
            sampled = group.sample(n=n, random_state=self.random_state)
            sampled_series.append(sampled)

        sampled_series_df = pd.concat(sampled_series, ignore_index=True)
        
        # OPTIMIZACIÓN: Usar merge en lugar de apply
        # Marcar series seleccionadas usando merge (mucho más rápido)
        df_marked = df.merge(
            sampled_series_df[pair_col].assign(_selected=True),
            on=pair_col,
            how='left'
        )
        df_marked['_selected'] = df_marked['_selected'].fillna(False)

        for i in range(self.test_months):
            test_date = max_date - i
            
            # Test: todas las series en test_date
            test_idx = np.where(df_marked['date_id'] == test_date)[0].tolist()

            # Train: solo series seleccionadas Y fechas anteriores
            train_idx = np.where(
                (df_marked['_selected'] == True) & 
                (df_marked['date_id'] < test_date-1)
            )[0].tolist()

            yield train_idx, test_idx

            
class SimpleLastDateSplitter(BaseCrossValidator):
    """Split: test = date_id máximo, train = resto. Sin copias innecesarias."""
    def get_n_splits(self, X=None, y=None, groups=None):
        return 1

    def split(self, X, y=None, groups=None):
        # No copies, solo uso la referencia
        max_date = X['date_id'].max()
        test_mask = X['date_id'] == max_date
        
        # Obtener posiciones enteras (para .iloc) en lugar de índices del DataFrame
        test_idx = np.where(test_mask)[0]
        train_idx = np.where(~test_mask)[0]
        
        yield train_idx, test_idx

In [9]:
# Reemplazo de inf por nan
df.replace([np.inf, -np.inf], np.nan, inplace=True)



# Transformar object a category (menos claves)
columns = df.select_dtypes(include=["object"]).columns.tolist()
for col in columns:
    if col not in ["product_id", "customer_id", "date_id"]:
        df[col] = df[col].astype("category")

# Eliminar columnas datetime innecesarias
datetime_cols = df.select_dtypes(include=["datetime"]).columns.tolist()
for col in datetime_cols:
    if col != "date_id":
        df.drop(columns=[col], inplace=True)

# Crear splitter con 20% y 2 meses

# Columnas a dropear
drop_cols = ["fecha", "target_scaled", "target", "date_id"]


class CustomMetric:
    def __init__(self, test_df, tn_std, product_ids):
        # Merge tn_std a test_df para alinear y guardar el resultado
        df_eval = test_df[["product_id", "customer_id", "target"]].copy()
        df_eval = df_eval.merge(
            tn_std[["product_id", "customer_id", "tn_std"]],
            on=["product_id", "customer_id"],
            how="left"
        )
        self.df_eval = df_eval
        self.product_ids = set(product_ids)

    def __call__(self, predt, dtrain):
        df_eval = self.df_eval.copy()
        df_eval["predictions"] = predt * df_eval["tn_std"]
        mask = df_eval["product_id"].isin(self.product_ids)
        df_grouped = df_eval.loc[mask].groupby("product_id", as_index=False)[["predictions", "target"]].sum()
        total_error = np.sum(np.abs(df_grouped["predictions"] - df_grouped["target"])) / np.sum(df_grouped["target"])
        return "total_error", total_error

In [10]:
import optuna
import numpy as np
import xgboost as xgb
#splitter = SimpleLastDateSplitter()
splitter = CustomTimeSeriesSplitter(subsample_prop=0.1, test_months=2, random_state=42)
def objective(trial):
    total_errors = []
    num_iterations_list = []

    # Hiperparámetros a optimizar
    params = {
        'objective': 'reg:tweedie',
        'tree_method': 'gpu_hist',
        'verbosity': 0,
        'sampling_method': 'uniform',  # Aproximación de extra_trees
        'tweedie_variance': trial.suggest_float("tweedie_variance", 1.1, 1.9),
        'learning_rate': trial.suggest_float("learning_rate", 0.01, 0.1),
        'max_depth': trial.suggest_int("max_depth", 3, 64),
        'lambda': trial.suggest_float("lambda", 0.0, 10.0),
        'alpha': trial.suggest_float("alpha", 0.0, 10.0),
        'min_child_weight': trial.suggest_int("min_child_weight", 1, 100),
        'subsample': trial.suggest_float("subsample", 0.5, 1.0),
        'colsample_bytree': trial.suggest_float("colsample_bytree", 0.3, 1.0),
    }

    for fold, (train_idx, test_idx) in enumerate(splitter.split(df)):
        train_df = df.iloc[train_idx].copy()
        test_df = df.iloc[test_idx].copy()
        train_df = train_df.groupby(["product_id", "customer_id"]).filter(lambda x: len(x) >= 12)

        print(f"Train shape: {train_df.shape}, Test shape: {test_df.shape}, Fold: {fold + 1}")

        X_train = train_df.drop(columns=[col for col in drop_cols if col in train_df.columns])
        y_train = train_df["target_scaled"]
        w_train = np.log1p(train_df["tn"]).clip(lower=1)

        X_test = test_df.drop(columns=[col for col in drop_cols if col in test_df.columns])
        y_test = test_df["target_scaled"]

        dtrain = xgb.DMatrix(X_train, label=y_train, weight=w_train, enable_categorical=True)
        dtest = xgb.DMatrix(X_test, label=y_test, enable_categorical=True)

        evals_result = {}
        model = xgb.train(
            params,
            dtrain=dtrain,
            num_boost_round=9999,
            evals=[(dtest, "eval")],
            early_stopping_rounds=int(400 + 4 / params["learning_rate"]),
            evals_result=evals_result,
            verbose_eval=100,
        )

        # get best iteration total error
        total_error = evals_result["eval"]["total_error"][-1]

        total_errors.append(total_error)
        num_iterations_list.append(model.best_iteration)

    avg_error = np.mean(total_errors)
    trial.set_user_attr("avg_num_iterations", np.mean(num_iterations_list))
    return avg_error

# Ejecutar Optuna
study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=24),
    study_name="exp_pipeline_xgb_gpu",
    storage="sqlite:///optuna_study.db",
    load_if_exists=True,
)
study.optimize(objective, n_trials=30)

# Mejor conjunto de parámetros

/home/fede/programacion/labo3/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2025-07-13 14:24:18,443] Using an existing study with name 'exp_pipeline_xgb_gpu' instead of creating a new one.
/tmp/ipykernel_355687/1029116073.py:46: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_marked['_selected'] = df_marked['_selected'].fillna(False)


Train shape: (67228, 1070), Test shape: (23998, 1070), Fold: 1
[0]	eval-tweedie-nloglik@1.5:3.05878
[100]	eval-tweedie-nloglik@1.5:2.51091
[200]	eval-tweedie-nloglik@1.5:2.56137
[300]	eval-tweedie-nloglik@1.5:2.61032
[400]	eval-tweedie-nloglik@1.5:2.66171
[500]	eval-tweedie-nloglik@1.5:2.71192


[W 2025-07-13 14:26:12,837] Trial 7 failed with parameters: {'tweedie_variance': 1.8680138426687347, 'learning_rate': 0.07295608449546184, 'max_depth': 64, 'lambda': 2.2006729978285176, 'alpha': 3.61056353964058, 'min_child_weight': 74, 'subsample': 0.9982278625445484, 'colsample_bytree': 0.5214428844534258} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/home/fede/programacion/labo3/.venv/lib/python3.12/site-packages/optuna/study/_optimize.py", line 197, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/tmp/ipykernel_355687/2381700294.py", line 44, in objective
    model = xgb.train(
            ^^^^^^^^^^
  File "/home/fede/programacion/labo3/.venv/lib/python3.12/site-packages/xgboost/core.py", line 726, in inner_f
    return func(**kwargs)
           ^^^^^^^^^^^^^^
  File "/home/fede/programacion/labo3/.venv/lib/python3.12/site-packages/xgboost/training.py", line 181, in train
    bst.update(dt

KeyboardInterrupt: 

In [ ]:
best_params = study.best_params
best_params.update({
    'objective': 'reg:tweedie',
    'tree_method': 'gpu_hist',
    'verbosity': 0,
    'sampling_method': 'uniform',
})

avg_best_iter = int(study.best_trial.user_attrs["avg_num_iterations"])
print(f"🔍 Mejor total_error: {study.best_value:.5f}")
print(f"🏁 Mejor iteración promedio: {avg_best_iter}")
print(f"📊 Mejor conjunto de parámetros: {best_params}")

# Entrenar modelo final
final_df = df.groupby(["product_id", "customer_id"]).filter(lambda x: len(x) >= 12)
X_final = final_df.drop(columns=[col for col in drop_cols if col in final_df.columns])
y_final = final_df["target_scaled"]
w_final = np.log1p(final_df["tn"]).clip(lower=1)
dtrain_final = xgb.DMatrix(X_final, label=y_final, weight=w_final)

final_model = xgb.train(
    best_params,
    dtrain=dtrain_final,
    num_boost_round=avg_best_iter,
    verbose_eval=10
)